In [10]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [8]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [11]:
system_prompt = """Act as a Senior Data Architect with extensive experience 
in designing stream data processing projects in both On-Premise and Google Cloud Platform."""

user_prompt = """
I am planning to design a process to parse streaming json data from Kafka to BigQuery. 
I need to perform a few complex transformations on the data before publishing in BigQuery.
What are different options available for me and analyse Pros and Cons of each option and finally provide a recommendation.
"""

In [15]:
messages = [
    {"role":"system", "content": system_prompt},
    {"role":"user", "content": user_prompt}
]

In [16]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)

In [17]:
print(response.choices[0].message.content)

Short answer / recommendation
- If you are running on Google Cloud and want a managed, scalable, low-ops solution for complex, stateful streaming transforms: use Apache Beam on Cloud Dataflow (KafkaIO) to read from Kafka, implement transforms in Beam, and write to BigQuery using the BigQuery Storage Write API or staged file-loads to GCS depending on latency/duplication needs.
- If you must stay on-prem or prefer Kafka-native stream processing: use Apache Flink or Kafka Streams (ksqlDB for SQL-style logic) to do the transforms, then sink to BigQuery by staging Parquet/Avro to GCS + load jobs or by calling BigQuery Storage Write API over secure connectivity.
- If transforms are light/simple and you prefer low engineering effort: use a managed BigQuery Sink (Confluent/Google connector) with any necessary SMTs/ksqlDB upstream.

Below I list the main implementation options, their pros/cons, typical use-cases and a recommended choice matrix to help decide.

Key requirements to think about up